In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_theme()

saveEMC = False
subBg = False

In [ ]:
emc_file = 'emc/protein_water_ds_4x_0001/data_1M/prot_only_4/output_130.h5'

with h5py.File(emc_file, "r") as f_ptr:
    I_emc = np.squeeze(f_ptr['intens'][:])
    W_emc = np.squeeze(f_ptr['inter_weight'][:])
    scale_emc = f_ptr['scale'][:]
    llk_emc = f_ptr['likelihood'][:]

I_emc = I_emc[:-1,:-1,:-1]
W_emc = W_emc[:-1,:-1,:-1]

if subBg:
    emc_bg_file = 'emc/water_only_ds_4x_0001/data_1M/wat_only_0/output_120.h5'
    with h5py.File(emc_bg_file,'r') as f_bg:
        I_emc_bg = np.squeeze(f_bg['intens'][:])
        scale_bg = f_bg['scale'][:]
        llk_bg = f_bg['likelihood'][:]

    I_emc_bg = I_emc_bg[:-1,:-1,:-1]
    frame = I_emc - I_emc_bg
else:
    frame = I_emc

center = frame.shape[0]//2
mask_emc = W_emc.astype(np.bool_)

In [ ]:
recon_intens = frame.copy()
recon_mask = mask_emc.copy()

corr_model_fname = input('Enter name of EMC model to save')
corr_model = recon_intens
corr_model[~recon_mask] = np.nan

fig_handle = plt.figure(constrained_layout = True, dpi = 170)
fig_handle.patch.set_facecolor('gray')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
emc_slice = 374//2

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(corr_model[:,emc_slice,:],vmin=0,vmax=0.1,cmap='cividis')
plt.xticks([])
plt.yticks([])
minv, maxv = im_0.get_clim()
c_bar_0 = plt.colorbar(im_0, ax=ax_0,shrink=0.42) 
c_bar_0.set_ticks([minv,maxv*0.5,maxv])

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
plt.imshow(corr_model[emc_slice,:,:],vmin=0,vmax=0.1,cmap='cividis')
plt.xticks([])
plt.yticks([])

ax_2 = fig_handle.add_subplot(spec_handle[0,2])
plt.imshow(corr_model[:,:,emc_slice],vmin=0,vmax=0.1,cmap='cividis')
plt.xticks([])
plt.yticks([]);

if saveEMC:
    with h5py.File('./' + corr_model_fname + "_corr.h5", mode="a") as handle:
        handle["intensity"] = corr_model